# PCA, NMF and anomaly detection — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D014_01_Anomaly_Detection_Fundamentals.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Anomaly Detection Fundamentals

## Learning Objectives
By the end of this notebook, you will:
- Understand what anomaly detection is and its applications
- Know the different types of anomalies
- Apply statistical methods (Z-score, IQR) for outlier detection
- Use Isolation Forest for anomaly detection
- Apply Local Outlier Factor (LOF)
- Understand One-Class SVM for novelty detection

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# ═══ CHECKPOINT — "What is Anomaly Detection?" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
plt.rcParams['figure.figsize'] = (10, 6)

print("Checkpoint restored. Ready for: What is Anomaly Detection?")

## 1. What is Anomaly Detection?

**Anomaly Detection** (also called outlier detection) is the identification of data points, observations, or patterns that deviate significantly from the expected behavior.

### Real-World Applications

| Domain | Application |
|--------|-------------|
| **Finance** | Credit card fraud detection |
| **Healthcare** | Disease outbreak detection |
| **Manufacturing** | Defect detection in products |
| **Cybersecurity** | Network intrusion detection |
| **IoT** | Sensor malfunction detection |
| **E-commerce** | Fake review detection |

### Types of Anomalies

1. **Point Anomalies**: A single data point is anomalous (e.g., unusual transaction amount)
2. **Contextual Anomalies**: Anomalous in a specific context (e.g., high temperature in winter)
3. **Collective Anomalies**: A collection of data points is anomalous together (e.g., network attack pattern)

In [ ]:
# TODO: compute `n_normal`, `normal_data`, `n_anomalies`
# Name your result: n_normal, normal_data, n_anomalies, anomalies

## 2. Statistical Methods

### 2.1 Z-Score Method

The Z-score measures how many standard deviations a point is from the mean.

$$z = \frac{x - \mu}{\sigma}$$

**Rule of thumb**: Points with |z| > 3 are typically considered anomalies.

In [ ]:
# TODO: compute `z_scores`, `anomaly_mask`, `zscore_anomalies`
# Name your result: z_scores, anomaly_mask, zscore_anomalies

### 2.2 IQR (Interquartile Range) Method

The IQR method is more robust to extreme outliers than Z-score.

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Points outside these bounds are considered anomalies.

In [ ]:
# TODO: compute `Q1`, `Q3`, `IQR`
# Name your result: Q1, Q3, IQR, lower_bound

## 3. Isolation Forest

### The Idea

Isolation Forest isolates anomalies using random binary trees. The key insight:

**Anomalies are easier to isolate** (fewer splits needed) because they are rare and different.

### How It Works

1. Randomly select a feature
2. Randomly select a split value between min and max
3. Repeat until point is isolated
4. Anomalies have **shorter path lengths** in trees

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `contamination` | Expected proportion of anomalies | 0.01 - 0.1 |
| `n_estimators` | Number of trees | 100 (default) |
| `max_samples` | Samples per tree | 'auto' or 256 |

In [ ]:
# TODO: compute `iso_forest`, `n_estimators`, `contamination`
# Name your result: iso_forest, n_estimators, contamination, random_state

In [ ]:
# ═══ CHECKPOINT — "Local Outlier Factor (LOF)" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
plt.rcParams['figure.figsize'] = (10, 6)
n_normal = 300
normal_data = np.vstack([np.random.randn(n_normal // 2, 2) * 0.5 + [2, 2], np.random.randn(n_normal // 2, 2) * 0.5 + [-2, -2]])
n_anomalies = 20
anomalies = np.random.uniform(low=-6, high=6, size=(n_anomalies, 2))
X = np.vstack([normal_data, anomalies])
y_true = np.array([0] * n_normal + [1] * n_anomalies)
def detect_anomalies_zscore(data, threshold=3):
    """
    Detect anomalies using Z-score method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    threshold : float
        Z-score threshold for anomaly detection
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    z_scores = np.abs(stats.zscore(data))
    anomaly_mask = np.any(z_scores > threshold, axis=1)
    return anomaly_mask
zscore_anomalies = detect_anomalies_zscore(X, threshold=2.5)
def detect_anomalies_iqr(data, multiplier=1.5):
    """
    Detect anomalies using IQR method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    multiplier : float
        IQR multiplier (1.5 = outlier, 3 = extreme outlier)
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    Q1 = np.percentile(data, 25, axis=0)
    Q3 = np.percentile(data, 75, axis=0)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    anomaly_mask = np.any((data < lower_bound) | (data > upper_bound), axis=1)
    return anomaly_mask
iqr_anomalies = detect_anomalies_iqr(X, multiplier=2.0)
iso_forest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
iso_predictions = iso_forest.fit_predict(X)
iso_anomalies = iso_predictions == -1
anomaly_scores = iso_forest.decision_function(X)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=anomaly_scores, cmap='RdYlBu', alpha=0.7)

print("Checkpoint restored. Ready for: Local Outlier Factor (LOF)")

## 4. Local Outlier Factor (LOF)

### The Idea

LOF compares the local density of a point to the local densities of its neighbors.

**Anomalies have substantially lower density** than their neighbors.

### How It Works

1. Find k-nearest neighbors for each point
2. Calculate **local reachability density** (LRD) for each point
3. Compare LRD of a point to LRD of its neighbors
4. LOF > 1 indicates lower density than neighbors (potential anomaly)

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `n_neighbors` | Number of neighbors to consider | 20 (default) |
| `contamination` | Expected proportion of anomalies | 0.01 - 0.1 |

In [ ]:
# TODO: compute `lof`, `n_neighbors`, `contamination`
# Name your result: lof, n_neighbors, contamination, novelty

## 5. One-Class SVM

### The Idea

One-Class SVM learns a decision boundary that encompasses the "normal" data.

**Points outside the boundary are anomalies.**

### How It Works

1. Map data to high-dimensional space using kernel
2. Find hyperplane that separates data from origin with maximum margin
3. Points on the "wrong side" of the hyperplane are anomalies

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `nu` | Upper bound on fraction of anomalies | 0.01 - 0.1 |
| `kernel` | Kernel type | 'rbf' (default) |
| `gamma` | Kernel coefficient | 'scale' or 'auto' |

In [ ]:
# TODO: compute `oc_svm`, `kernel`, `nu` from `red`
# Name your result: oc_svm, kernel, nu, gamma

## 6. Comparing Methods

Let's compare all the methods on our sample data.

In [ ]:
# TODO: draw a `plt.subplots`
# Name your result: methods

In [ ]:
# TODO: compute `methods_results`, `precision`, `recall`
# Name your result: methods_results, precision, recall, f1

In [ ]:
# ═══ CHECKPOINT — "When to Use Each Method" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
plt.rcParams['figure.figsize'] = (10, 6)
n_normal = 300
normal_data = np.vstack([np.random.randn(n_normal // 2, 2) * 0.5 + [2, 2], np.random.randn(n_normal // 2, 2) * 0.5 + [-2, -2]])
n_anomalies = 20
anomalies = np.random.uniform(low=-6, high=6, size=(n_anomalies, 2))
X = np.vstack([normal_data, anomalies])
y_true = np.array([0] * n_normal + [1] * n_anomalies)
def detect_anomalies_zscore(data, threshold=3):
    """
    Detect anomalies using Z-score method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    threshold : float
        Z-score threshold for anomaly detection
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    z_scores = np.abs(stats.zscore(data))
    anomaly_mask = np.any(z_scores > threshold, axis=1)
    return anomaly_mask
zscore_anomalies = detect_anomalies_zscore(X, threshold=2.5)
def detect_anomalies_iqr(data, multiplier=1.5):
    """
    Detect anomalies using IQR method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    multiplier : float
        IQR multiplier (1.5 = outlier, 3 = extreme outlier)
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    Q1 = np.percentile(data, 25, axis=0)
    Q3 = np.percentile(data, 75, axis=0)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    anomaly_mask = np.any((data < lower_bound) | (data > upper_bound), axis=1)
    return anomaly_mask
iqr_anomalies = detect_anomalies_iqr(X, multiplier=2.0)
iso_forest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
iso_predictions = iso_forest.fit_predict(X)
iso_anomalies = iso_predictions == -1
anomaly_scores = iso_forest.decision_function(X)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=anomaly_scores, cmap='RdYlBu', alpha=0.7)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, novelty=False)
lof_predictions = lof.fit_predict(X)
lof_anomalies = lof_predictions == -1
lof_scores = lof.negative_outlier_factor_
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=lof_scores, cmap='RdYlBu', alpha=0.7)
oc_svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm_predictions = oc_svm.fit_predict(X)
svm_anomalies = svm_predictions == -1
xx, yy = np.meshgrid(np.linspace(-7, 7, 100), np.linspace(-7, 7, 100))
Z = oc_svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
methods = [('Ground Truth', y_true == 1), ('Z-Score', zscore_anomalies), ('IQR', iqr_anomalies), ('Isolation Forest', iso_anomalies), ('Local Outlier Factor', lof_anomalies), ('One-Class SVM', svm_anomalies)]
from sklearn.metrics import precision_score, recall_score, f1_score
methods_results = [('Z-Score', zscore_anomalies), ('IQR', iqr_anomalies), ('Isolation Forest', iso_anomalies), ('LOF', lof_anomalies), ('One-Class SVM', svm_anomalies)]
for name, predictions in methods_results:
    precision = precision_score(y_true, predictions)
    recall = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    print(f'{name:<20} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f}')

print("Checkpoint restored. Ready for: When to Use Each Method")

## 7. When to Use Each Method

| Method | Best For | Limitations |
|--------|----------|-------------|
| **Z-Score** | Simple, univariate data | Assumes normal distribution |
| **IQR** | Robust to extreme outliers | May miss subtle anomalies |
| **Isolation Forest** | General purpose, large datasets | Less effective on low-dimensional data |
| **LOF** | Datasets with varying density | Computationally expensive for large datasets |
| **One-Class SVM** | Small to medium datasets | Slow training, requires tuning |

### Decision Guide

```
Is your data high-dimensional?
├── Yes → Isolation Forest (fast, scalable)
└── No
    ├── Does density vary across data?
    │   ├── Yes → LOF (handles varying density)
    │   └── No → One-Class SVM or Isolation Forest
    └── Simple baseline needed?
        └── Yes → Z-Score or IQR (fast, interpretable)
```

## 8. Practical Example: Transaction Fraud Detection

In [ ]:
# TODO: compute `n_normal_txn`, `normal_amounts`, `normal_hour` from `is_fraud`
# Name your result: n_normal_txn, normal_amounts, normal_hour, normal_frequency

In [ ]:
# Prepare features
X_txn = transactions[['amount', 'hour', 'daily_count']].values
y_txn = transactions['is_fraud'].values

# Scale features
scaler = StandardScaler()
X_txn_scaled = scaler.fit_transform(X_txn)

# Apply Isolation Forest
iso_forest_txn = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
predictions = iso_forest_txn.fit_predict(X_txn_scaled)
detected_fraud = predictions == -1

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix

print("Isolation Forest Fraud Detection Results:")
print("=" * 50)
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_txn, detected_fraud))
print(f"\nClassification Report:")
print(classification_report(y_txn, detected_fraud, target_names=['Normal', 'Fraud']))

In [ ]:
# TODO: draw a `plt.subplots`
# Name your result: c

## 9. Exercise: Anomaly Detection on Customer Data

**Scenario**: You have customer data and need to identify unusual customers for fraud investigation.

**Tasks:**
1. Load and explore the data
2. Scale the features
3. Apply Isolation Forest with contamination=0.05
4. Apply LOF and compare results
5. Visualize the detected anomalies

In [ ]:
# Create sample customer data
np.random.seed(42)
n_customers = 500

customers = pd.DataFrame({
    'annual_income': np.random.normal(50000, 15000, n_customers),
    'spending_score': np.random.normal(50, 20, n_customers),
    'age': np.random.normal(40, 12, n_customers)
})

# Add some anomalies manually
anomaly_indices = [0, 5, 10, 20, 100]
customers.loc[0, :] = [200000, 100, 25]  # Very high income and spending
customers.loc[5, :] = [10000, 99, 70]    # Low income, very high spending
customers.loc[10, :] = [150000, 5, 30]   # High income, very low spending
customers.loc[20, :] = [45000, 95, 18]   # Young with unusually high spending
customers.loc[100, :] = [180000, 2, 65]  # High income, almost no spending

print("Customer Data Sample:")
print(customers.head(10))

# YOUR CODE HERE
# 1. Scale the features using StandardScaler
# 2. Apply IsolationForest with contamination=0.05
# 3. Apply LocalOutlierFactor with contamination=0.05
# 4. Compare which anomalies each method detects
# 5. Create a scatter plot showing the detected anomalies



In [ ]:
# ═══ CHECKPOINT — "Summary" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
plt.rcParams['figure.figsize'] = (10, 6)
n_normal = 300
normal_data = np.vstack([np.random.randn(n_normal // 2, 2) * 0.5 + [2, 2], np.random.randn(n_normal // 2, 2) * 0.5 + [-2, -2]])
n_anomalies = 20
anomalies = np.random.uniform(low=-6, high=6, size=(n_anomalies, 2))
X = np.vstack([normal_data, anomalies])
y_true = np.array([0] * n_normal + [1] * n_anomalies)
def detect_anomalies_zscore(data, threshold=3):
    """
    Detect anomalies using Z-score method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    threshold : float
        Z-score threshold for anomaly detection
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    z_scores = np.abs(stats.zscore(data))
    anomaly_mask = np.any(z_scores > threshold, axis=1)
    return anomaly_mask
zscore_anomalies = detect_anomalies_zscore(X, threshold=2.5)
def detect_anomalies_iqr(data, multiplier=1.5):
    """
    Detect anomalies using IQR method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    multiplier : float
        IQR multiplier (1.5 = outlier, 3 = extreme outlier)
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    Q1 = np.percentile(data, 25, axis=0)
    Q3 = np.percentile(data, 75, axis=0)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    anomaly_mask = np.any((data < lower_bound) | (data > upper_bound), axis=1)
    return anomaly_mask
iqr_anomalies = detect_anomalies_iqr(X, multiplier=2.0)
iso_forest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
iso_predictions = iso_forest.fit_predict(X)
iso_anomalies = iso_predictions == -1
anomaly_scores = iso_forest.decision_function(X)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=anomaly_scores, cmap='RdYlBu', alpha=0.7)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, novelty=False)
lof_predictions = lof.fit_predict(X)
lof_anomalies = lof_predictions == -1
lof_scores = lof.negative_outlier_factor_
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=lof_scores, cmap='RdYlBu', alpha=0.7)
oc_svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm_predictions = oc_svm.fit_predict(X)
svm_anomalies = svm_predictions == -1
xx, yy = np.meshgrid(np.linspace(-7, 7, 100), np.linspace(-7, 7, 100))
Z = oc_svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
methods = [('Ground Truth', y_true == 1), ('Z-Score', zscore_anomalies), ('IQR', iqr_anomalies), ('Isolation Forest', iso_anomalies), ('Local Outlier Factor', lof_anomalies), ('One-Class SVM', svm_anomalies)]
from sklearn.metrics import precision_score, recall_score, f1_score
methods_results = [('Z-Score', zscore_anomalies), ('IQR', iqr_anomalies), ('Isolation Forest', iso_anomalies), ('LOF', lof_anomalies), ('One-Class SVM', svm_anomalies)]
for name, predictions in methods_results:
    precision = precision_score(y_true, predictions)
    recall = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    print(f'{name:<20} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f}')
n_normal_txn = 1000
normal_amounts = np.random.exponential(50, n_normal_txn)
normal_hour = np.random.randint(8, 22, n_normal_txn)
normal_frequency = np.random.poisson(3, n_normal_txn)
n_fraud_txn = 30
fraud_amounts = np.random.uniform(500, 2000, n_fraud_txn)
fraud_hour = np.random.randint(0, 6, n_fraud_txn)
fraud_frequency = np.random.poisson(15, n_fraud_txn)
transactions = pd.DataFrame({'amount': np.concatenate([normal_amounts, fraud_amounts]), 'hour': np.concatenate([normal_hour, fraud_hour]), 'daily_count': np.concatenate([normal_frequency, fraud_frequency]), 'is_fraud': [0] * n_normal_txn + [1] * n_fraud_txn})
X_txn = transactions[['amount', 'hour', 'daily_count']].values
y_txn = transactions['is_fraud'].values
scaler = StandardScaler()
X_txn_scaled = scaler.fit_transform(X_txn)
iso_forest_txn = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
predictions = iso_forest_txn.fit_predict(X_txn_scaled)
detected_fraud = predictions == -1
from sklearn.metrics import classification_report, confusion_matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_customers = 500
customers = pd.DataFrame({'annual_income': np.random.normal(50000, 15000, n_customers), 'spending_score': np.random.normal(50, 20, n_customers), 'age': np.random.normal(40, 12, n_customers)})
anomaly_indices = [0, 5, 10, 20, 100]
customers.loc[0, :] = [200000, 100, 25]
customers.loc[5, :] = [10000, 99, 70]
customers.loc[10, :] = [150000, 5, 30]
customers.loc[20, :] = [45000, 95, 18]
customers.loc[100, :] = [180000, 2, 65]

print("Checkpoint restored. Ready for: Summary")

## Summary

### Key Takeaways:

1. **Anomaly detection** identifies unusual patterns in data
2. **Statistical methods** (Z-score, IQR) are simple but assume distributions
3. **Isolation Forest** is fast and works well on high-dimensional data
4. **LOF** is good for data with varying densities
5. **One-Class SVM** learns a boundary around normal data
6. **No single method is best** - choose based on your data characteristics

### Quick Reference:

```python
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# Isolation Forest
iso = IsolationForest(contamination=0.1)
predictions = iso.fit_predict(X)  # -1 = anomaly, 1 = normal

# Local Outlier Factor
lof = LocalOutlierFactor(contamination=0.1)
predictions = lof.fit_predict(X)

# One-Class SVM
svm = OneClassSVM(nu=0.1)
predictions = svm.fit_predict(X)
```

---

**Next:** [../05_Case_Studies/01_Customer_Segmentation.ipynb](../05_Case_Studies/01_Customer_Segmentation.ipynb)